## Start with downloading Parquet files

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
pyspark.__file__

'C:\\tmp\\spark-win11-gcp-bigquery\\.venv\\Lib\\site-packages\\pyspark\\__init__.py'

In [4]:
# Question 1: Install Spark and PySpark
# Install Spark
# Run PySpark
# Create a local spark session
# Execute spark.version.
spark.version
# '4.1.1'

'4.1.1'

In [8]:
# Get data we will be using the Yellow 2025-11 data from the official website:
# Run once and comment out after download:

# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

# about 70MB in size...

# Question 2: Yellow November 2025
# Read the November 2025 Yellow into a Spark Dataframe.

# Repartition the Dataframe to 4 partitions and save it to parquet.

# What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

# 6MB
# 25MB
# 75MB - about 70MB in size...
# 100MB

## Table schema and count records

In [9]:
# Question 3: Count records
# How many taxi trips were there on the 15th of November?

# Consider only trips that started on the 15th of November.

# 62,610
# 102,340
# 162,604 == 15th november count is 162_604
# 225,768

df = spark.read \
    .parquet('yellow_tripdata_2025-11.parquet')

In [14]:
df.schema # looks good:

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [15]:
# or even with better formatting -
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [23]:
# How many taxi trips were there on the 15th of November?
df.count() # total row count is 4_181_444
df.filter((df.tpep_pickup_datetime >= '2025-11-15 00:00:00') & (df.tpep_pickup_datetime < '2025-11-16 00:00:00')).count()
# 15th november count is 162_604


162604

## Spark SQL

In [25]:
# double-check myself with Spark SQL
# df.registerTempTable('trips_data') # deprecated
df.createOrReplaceTempView('trips_data') 

In [26]:
spark.sql("""
SELECT
    count(*)
FROM
    trips_data
""").show()
# total row count is 4_181_444 as above 

+--------+
|count(1)|
+--------+
| 4181444|
+--------+



In [29]:
spark.sql("""
SELECT count(*)
FROM trips_data
WHERE  tpep_pickup_datetime >= '2025-11-15 00:00:00' AND tpep_pickup_datetime < '2025-11-16 00:00:00'
-- LIMIT 10
""").show()
# 162_604 as above.

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [33]:
# Question 4: Longest trip
# What is the length of the longest trip in the dataset in hours?

# 22.7
# 58.2
# 90.6 == 90.65
# 134.5

spark.sql("""
SELECT 
       ROUND(((UNIX_TIMESTAMP(tpep_dropoff_datetime) - 
        UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600), 2) AS duration 
FROM trips_data
ORDER BY 1 DESC
LIMIT 1
""").show()
# 90.65

+--------+
|duration|
+--------+
|   90.65|
+--------+



In [36]:
# Question 5: User Interface
# Spark's User Interface which shows the application's dashboard runs on which local port?

# 80
# 443
# 4040 == http://localhost:4040/jobs/
# 8080

# Question 6: Least frequent pickup location zone
# Load the zone lookup data into a temp view in Spark:
# RUN ONCE and comment it out after:
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
# Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

# Governor's Island/Ellis Island/Liberty Island = 1
# Arden Heights = 1
# Rikers Island 
# Jamaica Bay
# If multiple answers are correct, select any

df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [37]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [40]:
# join 2 dataframes/tables:
df_zones.createOrReplaceTempView('zones') 

In [45]:
# Using the zone lookup data and the Yellow November 2025 data, 
# what is the name of the LEAST frequent pickup location Zone?

spark.sql("""
SELECT z.Zone, COUNT(*)
FROM trips_data t JOIN zones z
    ON z.LocationID = t.PULocationID
GROUP BY 1
ORDER BY 2
LIMIT 10
""").show()
# |Governor's Island...|       1|
# |Eltingville/Annad...|       1|
# |       Arden Heights|       1

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
|         Great Kills|       4|
| Green-Wood Cemetery|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
+--------------------+--------+



In [46]:
# THE END...